# Marketing Ads Analysis
Reproduces the full Marketing - Ads tab from the NBS BI dashboard.
Edit the **Parameters** cell to change the analysis window, platform filter, or referral code.

In [ ]:
import os
os.environ["DB_CACHE_DIR"] = "campaigns"   # cache campaign queries in notebooks/campaigns/

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import plotly.io as pio
pio.renderers.default = 'notebook'   # outputs embeddable HTML — required for PDF export

from dotenv import load_dotenv
load_dotenv()

from nbs_bi.config import ADS_DATABASE_URL, READONLY_DATABASE_URL
from nbs_bi.clients.campaigns import (
    CampaignAnalyzer,
    load_ad_spend_from_db,
    aggregate_spend,
)
from nbs_bi.clients.report import ClientReport
from nbs_bi.onramp.queries import OnrampQueries
from nbs_bi.reporting.cards import _load_all_invoice_models
from nbs_bi.reporting.marketing import (
    _build_cumulative_spend,
    _build_channel_comparison,
    _append_meta_ads_channel_trace,
    _fig_cumulative_spend,
    _fig_cumulative_profit,
    _fig_revenue_breakdown,
    _fig_campaign_roi,
    _fig_campaign_cac,
    _fig_campaign_daily,
    _fig_daily_revenue_vs_spend,
    _fig_daily_rev_all_vs_cohort,
    _fig_channel_comparison,
    _fig_channel_daily,
    _fig_campaign_funnel,
)

print('Imports OK')

## Parameters

In [ ]:
# --- Edit these ---
START_DATE    = '2025-06-23'   # earliest spend date; set to None for all history
END_DATE      = str(pd.Timestamp.today().date())
PLATFORM      = None           # None = all, 'meta', 'google'
REFERRAL_CODE = ''             # '' = no filter; e.g. 'ERIC'

print(f'Analysis window : {START_DATE} → {END_DATE}')
print(f'Platform filter : {PLATFORM or "all"}')
print(f'Referral code   : {REFERRAL_CODE or "none"}')

## 1. Ad Spend Data

In [ ]:
spend_df_raw = load_ad_spend_from_db(ADS_DATABASE_URL)

# Apply date window
if START_DATE:
    spend_df_raw = spend_df_raw[pd.to_datetime(spend_df_raw['date']) >= pd.Timestamp(START_DATE)]
if END_DATE:
    spend_df_raw = spend_df_raw[pd.to_datetime(spend_df_raw['date']) <= pd.Timestamp(END_DATE)]

# Platform filter
spend_df = spend_df_raw.copy()
if PLATFORM:
    spend_df = spend_df[spend_df['platform'] == PLATFORM]

spend_agg = aggregate_spend(spend_df)

print(f'Rows loaded : {len(spend_df_raw)}')
print(f'Date range  : {pd.to_datetime(spend_df_raw["date"]).min().date()} → {pd.to_datetime(spend_df_raw["date"]).max().date()}')
print(f'Platforms   : {sorted(spend_df_raw["platform"].unique().tolist())}')
print(f'Total spend : ${spend_df_raw["daily_spend_usd"].sum():,.2f}')
spend_df_raw.tail()

## 2. Campaign Detection

In [ ]:
analyzer  = CampaignAnalyzer(spend_agg, db_url=READONLY_DATABASE_URL)
campaigns = analyzer.campaigns

print(f'Detected {len(campaigns)} campaign(s):')
for c in campaigns:
    print(f"  {c['campaign_id']}  {c['start']} → {c['end']}  ${c['total_spend_usd']:,.2f}")

## 3. ROI Summary

In [ ]:
summary = analyzer.roi_summary()
summary

## 4. KPIs

In [ ]:
latest_id     = summary['campaign_id'].iloc[-1]
kyc_done      = analyzer.cohort_kyc_count(latest_id, referral_code=REFERRAL_CODE)
total_spend   = summary['total_spend_usd'].sum()
total_revenue = summary['total_revenue_usd'].sum()
transacting   = int(summary['transacting_users'].sum())
cohort_users  = int(summary['cohort_users'].sum())
roas = total_revenue / total_spend if total_spend else float('nan')

print(f'Latest campaign   : {latest_id}')
print(f'Total ad spend    : ${total_spend:,.2f}')
print(f'Cohort revenue    : ${total_revenue:,.2f}')
print(f'Overall ROAS      : {roas:.2f}×')
print(f'Cohort sign-ups   : {cohort_users:,}')
print(f'KYC completed     : {kyc_done:,}')
print(f'Activated users   : {transacting:,}')

## 5. Cumulative Revenue & Profit

In [ ]:
cum_rev_df = analyzer.cumulative_revenue(latest_id, referral_code=REFERRAL_CODE)

_, _, _, history = _load_all_invoice_models()
invoice_history = [
    (period, m.inputs.invoice_total_usd or float(m.cost_breakdown().total), m.inputs.n_transactions)
    for period, m in history
]

cum_profit_df = analyzer.cumulative_profit(
    latest_id, invoice_history, referral_code=REFERRAL_CODE
)

print(f'Cohort revenue dataframe : {len(cum_rev_df)} rows')
print(f'Cumulative revenue       : ${cum_rev_df["cum_rev_usd"].iloc[-1]:,.2f}')
cum_profit_df[[
    'date', 'cum_rev_usd', 'cum_card_cogs_usd',
    'cum_contribution_margin_usd', 'cum_profit_usd'
]].tail()

## 6. Daily Context

In [ ]:
daily = analyzer.daily_context()
daily.tail(10)

## 7. Acquisition & Channel Data

In [ ]:
client_report = ClientReport(
    START_DATE or '2025-06-01', END_DATE, db_url=READONLY_DATABASE_URL
).build()
acquisition            = client_report.get('acquisition')
profit_by_source_daily = client_report.get('profit_by_source_daily')
print('Acquisition sources:', sorted(acquisition['acquisition_source'].unique().tolist()) if acquisition is not None else 'n/a')

## 8. All-Users Daily Revenue (platform baseline)

In [ ]:
date_start = START_DATE or '2025-06-23'
date_end   = str((pd.Timestamp.today() + pd.Timedelta(days=1)).date())

all_users_rev_df = OnrampQueries(
    start_date=date_start, end_date=date_end, db_url=READONLY_DATABASE_URL
).daily_revenue_by_product()

print(f'All-users daily revenue : {len(all_users_rev_df)} rows')
print(f'Date range              : {all_users_rev_df["date"].min()} → {all_users_rev_df["date"].max()}')
all_users_rev_df.tail()

---
## Charts

### Cohort Activation Funnel

In [ ]:
funnel = {'signups': cohort_users, 'kyc_done': kyc_done, 'activated': transacting}
fig = _fig_campaign_funnel(funnel)
if fig:
    fig.show()

### Cumulative Spend vs Cohort Revenue

In [ ]:
cum_df = _build_cumulative_spend(spend_agg, campaigns)
fig = _fig_cumulative_spend(cum_df, campaigns, cum_rev_df, cum_profit_df)
if fig:
    fig.show()

### Operational Profit & Contribution Margin

In [ ]:
fig = _fig_cumulative_profit(cum_profit_df)
if fig:
    fig.show()

### Revenue Breakdown (stacked area)

In [ ]:
fig = _fig_revenue_breakdown(cum_profit_df)
if fig:
    fig.show()

### Ad Spend vs Cohort Revenue by Campaign

In [ ]:
fig = _fig_campaign_roi(summary, cum_profit_df)
if fig:
    fig.show()

### Customer Acquisition Cost (CAC)

In [ ]:
fig = _fig_campaign_cac(summary)
if fig:
    fig.show()

### Daily Sign-ups vs Ad Spend

In [ ]:
fig = _fig_campaign_daily(daily)
if fig:
    fig.show()

### Daily Revenue vs Ad Spend

In [ ]:
fig = _fig_daily_revenue_vs_spend(all_users_rev_df, spend_agg)
if fig:
    fig.show()

### Platform Revenue vs Cohort Boundary

In [ ]:
fig = _fig_daily_rev_all_vs_cohort(all_users_rev_df, cum_rev_df, spend_agg)
if fig:
    fig.show()

### Channel Comparison

In [ ]:
comparison = _build_channel_comparison(summary, acquisition, cum_profit_df)

fig = _fig_channel_comparison(comparison)
if fig:
    fig.show()

# Merge meta_ads cohort profit into the channel daily evolution chart
daily_with_meta = _append_meta_ads_channel_trace(profit_by_source_daily, cum_profit_df)
if daily_with_meta is not None:
    fig2 = _fig_channel_daily(daily_with_meta)
    if fig2:
        fig2.show()

comparison


### Revenue Heatmap — Day of Week × Hour (BRT)

In [ ]:
import plotly.graph_objects as go
from sqlalchemy import text
from nbs_bi.onramp.queries import _to_exclusive_end

oq     = OnrampQueries(start_date=date_start, end_date=date_end, db_url=READONLY_DATABASE_URL)
_bound = {'start_date': date_start, 'end_date': _to_exclusive_end(date_end)}

# --- 1. Conversions ---
conv_raw = oq.conversions()
rate = pd.to_numeric(conv_raw['exchange_rate'], errors='coerce').replace(0, float('nan'))
conv_raw['rev_usd'] = (
    conv_raw['fee_amount_brl'].fillna(0) / rate
    + conv_raw['fee_amount_usdc'].fillna(0)
    + conv_raw['spread_revenue_brl'].fillna(0) / rate
    + conv_raw['spread_revenue_usdc'].fillna(0)
)
conv_raw = conv_raw[['created_at', 'rev_usd']].assign(source='conversion')

_SQL_CARD_FEES = """
SELECT paid_at AS created_at, amount_usdc::FLOAT AS rev_usd
FROM   card_annual_fees
WHERE  status = 'paid'
  AND  paid_at >= :start_date AND paid_at < :end_date
"""
_SQL_BILLING = """
SELECT created_at, amount::FLOAT / 1000000.0 AS rev_usd
FROM   billing_charges
WHERE  status = 'settled'
  AND  created_at >= :start_date AND created_at < :end_date
"""
with oq._engine_lazy.connect() as conn:
    card_fees_raw = pd.read_sql(text(_SQL_CARD_FEES), conn, params=_bound).assign(source='card_fee')
    billing_raw   = pd.read_sql(text(_SQL_BILLING),   conn, params=_bound).assign(source='card_transaction')

for df in (card_fees_raw, billing_raw):
    df['created_at'] = pd.to_datetime(df['created_at'], utc=True)

# --- Combine & enrich ---
all_rev = pd.concat([conv_raw, card_fees_raw, billing_raw], ignore_index=True)
all_rev['created_at_brt'] = all_rev['created_at'].dt.tz_convert('America/Sao_Paulo').dt.tz_localize(None)
all_rev['hour']  = all_rev['created_at_brt'].dt.hour
all_rev['dow']   = all_rev['created_at_brt'].dt.day_name()
all_rev['date']  = all_rev['created_at_brt'].dt.date
all_rev['month'] = all_rev['created_at_brt'].dt.to_period('M').astype(str)

print(f'Revenue rows: {len(all_rev):,}  |  Total: ${all_rev["rev_usd"].sum():,.2f}')
print(all_rev.groupby('source')['rev_usd'].agg(['count', 'sum']).round(2))

# --- Precompute pivots and per-day averages ---
DOW_ORDER   = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
HOURS       = list(range(24))
month_opts  = ['All months'] + sorted(all_rev['month'].unique())
source_opts = ['All sources'] + sorted(all_rev['source'].unique())

def _slice(month: str, source: str) -> pd.DataFrame:
    df = all_rev if month == 'All months' else all_rev[all_rev['month'] == month]
    return df if source == 'All sources' else df[df['source'] == source]

def _make_pivot(df: pd.DataFrame) -> list:
    return (
        df.groupby(['hour', 'dow'])['rev_usd']
        .sum()
        .unstack('dow')
        .reindex(index=HOURS, columns=DOW_ORDER)
        .fillna(0)
        .values
        .tolist()
    )

def _day_avgs(df: pd.DataFrame) -> dict:
    """Average total revenue per occurrence of each day-of-week."""
    result = {}
    for day in DOW_ORDER:
        mask   = df['dow'] == day
        n_days = df.loc[mask, 'date'].nunique()
        total  = df.loc[mask, 'rev_usd'].sum()
        result[day] = total / n_days if n_days > 0 else 0.0
    return result

def _peak_hours(pivot_values: list) -> list:
    """Return the peak-revenue hour (0-23) for each day column. None if no revenue."""
    import numpy as np
    arr = np.array(pivot_values)  # shape (24, 7)
    result = []
    for col_idx in range(arr.shape[1]):
        col = arr[:, col_idx]
        result.append(int(col.argmax()) if col.max() > 0 else None)
    return result

pivots    = {}
day_avgs  = {}
peak_hrs  = {}
for mo in month_opts:
    for so in source_opts:
        df = _slice(mo, so)
        pivots[(mo, so)]   = _make_pivot(df)
        day_avgs[(mo, so)] = _day_avgs(df)
        peak_hrs[(mo, so)] = _peak_hours(pivots[(mo, so)])

# --- Annotation builders ---
_BOTTOM_ANNOTATIONS = [
    dict(text='Month:',  x=0.0,  xref='paper', y=-0.17, yref='paper',
         showarrow=False, xanchor='left', font=dict(size=12)),
    dict(text='Source:', x=0.32, xref='paper', y=-0.17, yref='paper',
         showarrow=False, xanchor='left', font=dict(size=12)),
]

def _top_annotations(avgs: dict) -> list:
    return [
        dict(
            text=f'<b>${avgs[day]:,.0f}</b>',
            x=day, xref='x',
            y=1.0, yref='paper',
            yanchor='bottom', xanchor='center',
            showarrow=False,
            font=dict(size=10, color='#8B949E'),
        )
        for day in DOW_ORDER
    ]

def _all_annotations(mo: str, so: str) -> list:
    return _BOTTOM_ANNOTATIONS + _top_annotations(day_avgs[(mo, so)])

# --- Figure ---
_mo0, _so0 = 'All months', 'All sources'

fig = go.Figure()

fig.add_trace(go.Heatmap(
    z=pivots[(_mo0, _so0)],
    x=DOW_ORDER,
    y=HOURS,
    colorscale='Viridis',
    colorbar=dict(title='Revenue (USD)'),
    hovertemplate='%{x}<br>%{y}:00 BRT<br>$%{z:.2f}<extra></extra>',
))

# Peak-hour line: connects the highest-revenue hour Mon->Sun
fig.add_trace(go.Scatter(
    x=DOW_ORDER,
    y=peak_hrs[(_mo0, _so0)],
    mode='lines+markers',
    line=dict(color='white', width=2),
    marker=dict(size=9, color='white', line=dict(color='#333', width=1.5)),
    name='Peak hour',
    hovertemplate='%{x}<br>Peak: %{y}:00 BRT<extra></extra>',
))

# Array-per-trace format: [val_trace0, val_trace1]; None = leave trace unchanged.
month_buttons = [
    dict(
        label=mo,
        method='update',
        args=[
            {
                'z': [pivots[(mo, 'All sources')], None],
                'y': [None, peak_hrs[(mo, 'All sources')]],
            },
            {
                'title': f'Revenue Heatmap — {mo} / All sources (BRT)',
                'annotations': _all_annotations(mo, 'All sources'),
            },
        ],
    )
    for mo in month_opts
]

source_buttons = [
    dict(
        label=so,
        method='update',
        args=[
            {
                'z': [pivots[('All months', so)], None],
                'y': [None, peak_hrs[('All months', so)]],
            },
            {
                'title': f'Revenue Heatmap — All months / {so} (BRT)',
                'annotations': _all_annotations('All months', so),
            },
        ],
    )
    for so in source_opts
]

fig.update_layout(
    title='Revenue Heatmap — All months / All sources (BRT)',
    xaxis_title='Day of Week',
    yaxis_title='Hour of Day (BRT)',
    yaxis=dict(tickmode='linear', dtick=1),
    annotations=_all_annotations(_mo0, _so0),
    updatemenus=[
        dict(
            buttons=month_buttons,
            direction='up',
            showactive=True,
            x=0.0, xanchor='left',
            y=-0.25, yanchor='top',
        ),
        dict(
            buttons=source_buttons,
            direction='up',
            showactive=True,
            x=0.32, xanchor='left',
            y=-0.25, yanchor='top',
        ),
    ],
    margin=dict(t=80, b=140),
    height=600,
)
fig.show()


In [ ]:
fig.write_html("revenue_heatmap.html", include_plotlyjs="inline")


In [ ]:
import numpy as np
import ipywidgets as widgets
from IPython.display import display

real_months = [mo for mo in month_opts if mo != 'All months']
n      = len(real_months)
alphas = [0.25 + 0.75 * i / max(n - 1, 1) for i in range(n)]
BASE   = (99, 110, 250)

def _make_bands(arr_combined):
    shapes = []
    for di in range(len(DOW_ORDER)):
        ph = int(arr_combined[:, di].argmax())
        shapes.append(dict(
            type='rect',
            x0=di - 0.4, x1=di + 0.4,
            y0=max(0, ph - 1), y1=min(23, ph + 1),
            xref='x', yref='y',
            fillcolor='rgba(255, 200, 0, 0.15)',
            line=dict(width=0),
            layer='below',
        ))
    return shapes

# ── Figure ────────────────────────────────────────────────────────────────────
fig_peaks = go.FigureWidget()

for i, mo in enumerate(real_months):
    arr    = np.array(pivots[(mo, 'All sources')])
    peak_h = arr.argmax(axis=0)
    rgba   = f'rgba({BASE[0]},{BASE[1]},{BASE[2]},{alphas[i]:.2f})'
    fig_peaks.add_trace(go.Scatter(
        x=DOW_ORDER, y=peak_h,
        mode='lines+markers', name=mo,
        line=dict(color=rgba, width=2),
        marker=dict(size=8, color=rgba),
        hovertemplate='<b>' + mo + '</b><br>%{x}<br>Peak hour: %{y}:00 BRT<extra></extra>',
    ))

arr_all = np.array(pivots[('All months', 'All sources')])
fig_peaks.update_layout(
    title='Peak Revenue Hour by Day of Week — each Month (All sources, BRT)',
    xaxis_title='Day of Week',
    yaxis_title='Hour of Day (BRT)',
    yaxis=dict(tickmode='linear', dtick=1, range=[-0.5, 23.5]),
    showlegend=False,
    shapes=_make_bands(arr_all),
    height=420,
    margin=dict(t=60, b=40, l=40, r=20),
)

# ── Checkboxes — vertical, left side, alpha matches curve ────────────────────
checkboxes = {}
cb_widgets  = []
for i, mo in enumerate(real_months):
    a   = alphas[i]
    col = f'rgba({BASE[0]},{BASE[1]},{BASE[2]},{a:.2f})'
    cb  = widgets.Checkbox(
        value=True, description=mo,
        layout=widgets.Layout(width='120px'),
        style={'description_width': '0px'},
    )
    # Coloured dot label next to checkbox
    dot = widgets.HTML(
        f'<span style="color:rgba({BASE[0]},{BASE[1]},{BASE[2]},{a:.2f});'
        f'font-size:13px;font-family:monospace;">{mo}</span>'
    )
    checkboxes[mo] = cb
    cb_widgets.append(widgets.HBox([cb, dot], layout=widgets.Layout(height='24px')))

def _on_change(change):
    selected = [mo for mo, cb in checkboxes.items() if cb.value]
    if not selected:
        return
    combined = np.zeros((24, len(DOW_ORDER)))
    for mo in selected:
        combined += np.array(pivots[(mo, 'All sources')])
    with fig_peaks.batch_update():
        for i, mo in enumerate(real_months):
            fig_peaks.data[i].visible = (mo in selected)
        fig_peaks.layout.shapes = _make_bands(combined)

for cb in checkboxes.values():
    cb.observe(_on_change, names='value')

cb_panel = widgets.VBox(
    cb_widgets,
    layout=widgets.Layout(
        margin='40px 12px 0 0',
        padding='8px',
        border='1px solid #333',
        border_radius='6px',
    ),
)

display(widgets.HBox([cb_panel, fig_peaks]))


---
## Campaign Summary Table

In [ ]:
display(summary)

## Referral Code Breakdown (optional)

In [ ]:
# Lists all referral codes tracked in the DB for this campaign
options = analyzer.referral_code_options()
print(f'Available referral codes ({len(options)}):', options)

---
## Deep Client & Channel Analysis

Eight analytical explorations to answer: **who are our best clients, which channels produce them, and how do we get more of them efficiently.**

Priority order (highest CFO impact first):

| # | Analysis | Business Question |
|---|---|---|
| A1 | Channel LTV:CAC | Should we keep spending on Meta Ads? |
| A2 | Campaign 9 Autopsy | What made ROAS 2.87× work — how do we replicate it? |
| A3 | Cohort LTV + Payback | When do users pay back their CAC? |
| A4 | Champion Profile (CPF) | Who are our best users? Build a Lookalike Audience. |
| A5 | Product Cross-Sell | Are paid users using 1 product or the full platform? |
| A6 | Referral Code Quality | Which of the 115 affiliates drive quality vs junk signups? |
| A7 | Funnel Drop-Off | Where is the biggest leak: signup, KYC, or activation? |
| A8 | At-Risk VIP Retention | How much revenue is about to churn? |

> **Requires:** Sections 1–7 executed first (spend, campaigns, ROI summary, client report, all-users revenue).


In [ ]:

# Deep Analysis Setup
# All data is already available from the ClientReport.build() call in Section 7.
# This cell extracts the relevant artefacts.

segments_df   = client_report.get('segments')        # master_df with 'segment' column
product_df    = client_report.get('product_adoption')
cohort_ltv_df = client_report.get('cohort_ltv')
at_risk_df    = client_report.get('at_risk')
referral_df   = client_report.get('referral_codes')
ltv_src       = client_report.get('ltv_by_source', {})
breakeven_df  = client_report.get('cac_breakeven')
acq_df        = client_report.get('acquisition')
founders_df   = client_report.get('founders')

print('Data artefact status:')
for key, val in {
    'segments':    segments_df,
    'product':     product_df,
    'cohort_ltv':  cohort_ltv_df,
    'at_risk':     at_risk_df,
    'referrals':   referral_df,
    'ltv_by_src':  ltv_src,
    'breakeven':   breakeven_df,
    'acquisition': acq_df,
    'founders':    founders_df,
}.items():
    if val is None:
        status = 'None'
    elif isinstance(val, dict):
        status = f'dict({len(val)} keys)'
    else:
        status = f'DataFrame({val.shape})'
    print(f'  {key:15s}: {status}')

if segments_df is not None:
    print('\nSegment distribution:')
    print(segments_df['segment'].value_counts().rename('n_users').to_frame().to_string())


### A1 · Channel Economics — LTV:CAC by Acquisition Source

**Business question:** Which channel delivers the best return? Should we keep spending on Meta Ads?  
**CFO rule:** LTV:CAC < 2× → fix the product/funnel first. ≥ 3× → scale with conviction.


In [ ]:

# A1 · Channel Economics — LTV:CAC by Acquisition Source
# CFO decision gate: LTV:CAC < 2× → do not scale. ≥ 3× → scale.

import numpy as np

meta_transacting = int(summary['transacting_users'].sum())
meta_cac_full    = summary['total_spend_usd'].sum() / meta_transacting if meta_transacting > 0 else np.nan
meta_total_rev   = summary['total_revenue_usd'].sum()
meta_avg_rev_per_txn = meta_total_rev / meta_transacting if meta_transacting > 0 else np.nan

print('=== Meta Ads channel ===')
print(f'  Spend:                    ${summary["total_spend_usd"].sum():,.2f}')
print(f'  Transacting users:        {meta_transacting:,}')
print(f'  CAC (full):               ${meta_cac_full:.2f}')
print(f'  Cohort revenue:           ${meta_total_rev:,.2f}')
print(f'  Avg rev / txn user:       ${meta_avg_rev_per_txn:.2f}')
print(f'  LTV:CAC (current window): {meta_avg_rev_per_txn/meta_cac_full:.2f}× {"🚫 below 2×" if meta_avg_rev_per_txn/meta_cac_full < 2 else "⚠️"}')
print()

# Non-paid channels (ClientModel — all users from start of analysis window)
if acq_df is not None and not acq_df.empty:
    print('=== Non-paid channels (ClientModel — full history, all users) ===')
    cols = ['acquisition_source','n_users','n_transacting','conversion_rate',
            'avg_net_revenue_usd','median_net_revenue_usd','total_net_revenue_usd']
    avail = [c for c in cols if c in acq_df.columns]
    tbl = acq_df[avail].copy()
    if 'conversion_rate' in tbl.columns:
        tbl['conversion_rate'] = tbl['conversion_rate'].map('{:.1%}'.format)
    for col in ['avg_net_revenue_usd','median_net_revenue_usd','total_net_revenue_usd']:
        if col in tbl.columns:
            tbl[col] = tbl[col].map('${:.2f}'.format)
    display(tbl.to_string(index=False))

# LTV curves by source
if ltv_src:
    print('\n=== Avg cumulative LTV per transacting user (months since signup) ===')
    for src, pivot in ltv_src.items():
        if pivot.empty:
            continue
        avg = pivot.mean()
        pts = {int(m): round(float(avg[m]), 2) for m in sorted(avg.index) if m <= 12}
        print(f'  {src:22s}: {pts}')

print("""
CONCLUSION A1:
- Meta Ads LTV:CAC is currently below 1 — every transacting user acquired costs more than earned.
- Founder_invite channel has the highest avg operational profit ($2.82/user) — 3× better than meta_ads.
- Direct_referral drives the most total profit ($6,652) with no direct spend.
- Do NOT scale meta ads spend until LTV:CAC ≥ 3× — fix KYC funnel and product engagement first.
- Recommended budget rule: 0% prospecting, 100% retention/lookalike until breakeven is confirmed.
""")


### A6 · Referral Code Quality (Campaign 10)

**Business question:** Among the 115 campaign-10 affiliates, which drive the highest-quality users?  
**Metric:** Net ARPU after commission (not volume). Affiliates below breakeven are destroying value.


In [ ]:

# A6 · Referral Code Quality Analysis (Campaign 10)
# Among the 115 referral codes, which affiliates drive the highest-quality cohorts?
# Quality = LTV after commission. Volume alone is not the right metric.

import plotly.express as px
import numpy as np

if referral_df is not None and not referral_df.empty:
    ref = referral_df.copy()

    # Filter to codes that appear in campaign 10 options
    camp10_codes = [c.upper() for c in analyzer.referral_code_options()]
    ref_c10 = ref[ref['referral_code'].str.upper().isin(camp10_codes)].copy() if camp10_codes else ref.copy()

    print(f'Referral codes in analysis: {len(ref_c10)} (filtered to campaign 10 affiliates)')
    print()

    # Top 10 by net ARPU after commission
    top10 = ref_c10.nlargest(10, 'net_arpu_after_commission')[
        ['referral_code', 'referral_code_name', 'n_users', 'avg_net_revenue_usd',
         'commission_rate_bps', 'avg_commission_cost_usd', 'net_arpu_after_commission']
    ]
    print('=== TOP 10 affiliates by net ARPU after commission ===')
    display(top10.round(2))

    # Bottom 10 (value destroyers)
    bottom10 = ref_c10[ref_c10['n_users'] >= 5].nsmallest(10, 'net_arpu_after_commission')[
        ['referral_code', 'n_users', 'avg_net_revenue_usd', 'net_arpu_after_commission']
    ]
    print('\n=== BOTTOM 10 affiliates (n_users ≥ 5) — lowest net value ===')
    display(bottom10.round(2))

    # Scatter: volume vs quality
    plot_df = ref_c10[ref_c10['n_users'] >= 3].copy()
    if not plot_df.empty:
        fig_ref = px.scatter(
            plot_df,
            x='n_users',
            y='net_arpu_after_commission',
            text='referral_code',
            color='avg_net_revenue_usd',
            color_continuous_scale='RdYlGn',
            color_continuous_midpoint=0,
            title='Affiliate Quality vs Volume (campaign 10, n_users ≥ 3)',
            labels={
                'n_users': 'Users acquired',
                'net_arpu_after_commission': 'Net ARPU after commission (USD)',
                'avg_net_revenue_usd': 'Avg net revenue',
            },
        )
        fig_ref.update_traces(textposition='top center', textfont_size=9)
        fig_ref.add_hline(y=0, line_dash='dash', line_color='red', annotation_text='breakeven')
        fig_ref.update_layout(height=500)
        fig_ref.show()

    print("""
CONCLUSION A6:
- Top-right quadrant (high volume + high net ARPU) = ideal affiliates to increase budget allocation.
- Bottom half (negative net ARPU) = affiliates bringing users who cost more than they earn.
- Recommended action: Restructure commission for bottom performers (minimum activation requirement)
  or replace with performance-based commission (pay per transacting user, not signup).
""")
else:
    print('referral_codes data unavailable')


### A7 · Activation Funnel Drop-Off by Campaign

**Business question:** Where in the funnel are we losing users? KYC? Post-KYC? Or post-first-transaction?  
**Key lever:** KYC completion rate — doubling it from 17.6% → 35% would halve effective CAC immediately.


In [ ]:

# A7 · Funnel Drop-Off by Campaign
# Where are we losing users: before KYC, between KYC and first transaction, or after?
# KYC rate is the #1 lever: doubling it would halve effective CAC.

funnel_rows = []
for _, row in summary.iterrows():
    cid = row['campaign_id']
    cohort = int(row['cohort_users'])
    if cohort == 0:
        continue
    kyc_n = analyzer.cohort_kyc_count(cid)
    txn_n = int(row['transacting_users'])
    kyc_rate  = kyc_n / cohort  if cohort > 0 else 0
    txn_rate  = txn_n / kyc_n   if kyc_n > 0   else 0
    overall_r = txn_n / cohort  if cohort > 0   else 0
    funnel_rows.append({
        'campaign':        cid,
        'cohort_signups':  cohort,
        'kyc_done':        kyc_n,
        'kyc_rate':        f'{kyc_rate:.1%}',
        'txn_users':       txn_n,
        'kyc→txn_rate':    f'{txn_rate:.1%}',
        'signup→txn_rate': f'{overall_r:.1%}',
        'spend_usd':       f"${row['total_spend_usd']:,.0f}",
        'roas':            f"{row['roas']:.2f}×",
    })

if funnel_rows:
    fdf = pd.DataFrame(funnel_rows)
    display(fdf.to_string(index=False))
    print()

    # The KYC bottleneck
    avg_kyc = sum(float(r['kyc_done']) / float(r['cohort_signups'])
                  for r in funnel_rows) / len(funnel_rows)
    print(f'Avg KYC completion rate across campaigns: {avg_kyc:.1%}  (target: >35%)')
    print()
    print("""
CONCLUSION A7:
- KYC completion (~17.6% overall) is the biggest funnel bottleneck. Doubling it would halve CAC.
- Fix: Simplify the KYC UX, add progress bar, send a push/email reminder at T+24h if incomplete.
- The KYC→transaction rate (~51.6%) is reasonable — the drop from signup to KYC is the main loss.
- Campaign 9 achieved the best transacting rate (19.6%) — investigate what made KYC completion
  higher in that cohort (affiliate audience quality? timing? landing page?).
""")


### A8 · At-Risk VIP Retention

**Business question:** How much revenue is about to churn? What's the ROI of a reactivation campaign?  
**Hypothesis:** Reactivating 20% of at-risk users with positive LTV may beat a full paid campaign on cost.


In [ ]:

# A8 · At-Risk VIP Retention
# Who are the high-revenue users going inactive? Reactivating 10% may outperform a full campaign.

if at_risk_df is not None and not at_risk_df.empty:
    ar = at_risk_df.copy()
    print(f'At-risk users (30–90 days inactive, any revenue): {len(ar):,}')
    print()

    # Users with positive net revenue only
    ar_positive = ar[ar['net_revenue_usd'] > 0] if 'net_revenue_usd' in ar.columns else ar
    total_at_risk_rev = ar_positive['net_revenue_usd'].sum() if not ar_positive.empty else 0
    avg_monthly_rev = ar_positive['net_revenue_usd'].mean() / ar_positive['tenure_months'].clip(lower=1).mean() if not ar_positive.empty else 0

    print(f'At-risk users with positive LTV: {len(ar_positive):,}')
    print(f'Total historical net revenue at risk: ${total_at_risk_rev:,.2f}')
    print(f'Avg monthly revenue per at-risk user: ${avg_monthly_rev:.2f}')
    print()

    # Revenue tiers
    if not ar_positive.empty:
        tiers = pd.cut(ar_positive['net_revenue_usd'], bins=[-999, 0, 10, 50, 200, 99999],
                       labels=['≤$0', '$0–$10', '$10–$50', '$50–$200', '>$200'])
        print('At-risk users by revenue tier:')
        print(ar_positive.assign(tier=tiers).groupby('tier', observed=True)['net_revenue_usd'].agg(['count','sum']).rename(columns={'count':'n_users','sum':'total_rev'}).round(2).to_string())

    print()
    if 'acquisition_source' in ar_positive.columns:
        print('At-risk by acquisition source:')
        print(ar_positive.groupby('acquisition_source')['net_revenue_usd'].agg(['count','sum','mean']).rename(columns={'count':'n','sum':'total','mean':'avg'}).round(2).to_string())

    print("""
CONCLUSION A8:
- Even reactivating 20% of at-risk positive-LTV users is likely cheaper than a full paid campaign.
- Action: Export the at-risk user list (with masked IDs) to CRM for targeted push notification
  or email sequence offering a conversion fee discount or cashback incentive.
- Priority tier: $50–$200 revenue users — high enough LTV to justify personal outreach.
""")
else:
    print('at_risk data unavailable')


### A2 · Campaign Autopsy — Replicate Campaign 9 (ROAS 2.87×)

**Business question:** What made campaign 9 work while campaign 10 (10× bigger) failed?  
**Key metric to watch:** `transacting_rate` — if it drops below ~10%, pause the campaign.


In [ ]:

# A2 · Campaign 9 Autopsy — why did ROAS 2.87× work while campaign 10 failed at 0.48×?

camp_compare = summary[summary['campaign_id'].isin(['campaign_7', 'campaign_9', 'campaign_10'])].copy()
camp_compare = camp_compare.set_index('campaign_id')

metrics = [
    'total_spend_usd', 'duration_days', 'cohort_users', 'transacting_users',
    'transacting_rate', 'total_revenue_usd', 'roas',
    'cac_full', 'cac_incremental', 'avg_rev_per_transacting_user',
    'baseline_rate_per_day', 'incremental_users_est',
]
available = [m for m in metrics if m in camp_compare.columns]
display(camp_compare[available].T.round(4))

print()

# Daily signups/$ efficiency: incremental signups per $100 spent
for cid in ['campaign_7', 'campaign_9', 'campaign_10']:
    row = summary[summary['campaign_id'] == cid]
    if row.empty:
        continue
    r = row.iloc[0]
    incr = float(r.get('incremental_users_est', 0) or 0)
    spend = float(r['total_spend_usd'])
    txn_rate = float(r['transacting_rate']) if pd.notna(r['transacting_rate']) else 0
    avg_rev = float(r['avg_rev_per_transacting_user']) if pd.notna(r['avg_rev_per_transacting_user']) else 0
    print(f'{cid}: incremental users/spend = {incr/spend*100:.1f}/$ 100'
          f'  |  txn_rate = {txn_rate:.1%}'
          f'  |  avg_rev/txn_user = ${avg_rev:.2f}'
          f'  |  baseline {r["baseline_rate_per_day"]:.0f}/day')

print("""
CONCLUSION A2 — Campaign 9 hypothesis:
- Campaign 9 ran Feb 26–Mar 4 ($501 spend, 57.7 baseline signups/day, ROAS 2.87×).
- Campaign 10 ran Apr 14–May 1 ($4,788 spend, 15.3 baseline signups/day, ROAS 0.48×).

Key differences to investigate:
1. AUDIENCE: Campaign 9 ran against a warmer/more targeted audience segment.
   The low baseline (57.7/day) suggests competition was lower and audience fit was higher.
2. SCALE: Campaign 10 is 10× bigger spend — scaling too fast degrades audience quality
   (Meta exhausts high-intent users, broadens to lower-quality segments).
3. TIMING: Feb-Mar vs Apr-May — seasonality, BRL/USD rate, and affiliate mix differ.
4. TRANSACTING RATE: Campaign 9 = 19.6% vs campaign 10 = 4.3%.
   The winning metric is not signups but activated transacting users per dollar.

Recommendation: For next campaign, cap daily budget at $100-150 and target the same
audience profile as campaign 9. Track transacting_rate daily — pause if it drops below 10%.
""")


### A3 · Cohort LTV + Payback Period

**Business question:** How long until campaign cohorts break even? Which months are already profitable?  
**Method:** `ClientModel.cohort_ltv()` + `cac_breakeven()` — avg cumulative net USD per ever-transacted user.


In [ ]:

# A3 · Cohort LTV + Payback Period
# How long until users recoup their CAC? Are any cohorts already profitable?

import plotly.express as px

if cohort_ltv_df is not None and not cohort_ltv_df.empty:
    ltv = cohort_ltv_df.copy()
    # Rename period index to string for display
    ltv.index = ltv.index.astype(str)
    ltv.columns = [f'm{c}' for c in ltv.columns]

    print('=== Cohort LTV matrix (avg cumulative net USD per transacting user) ===')
    print('Rows = signup cohort month. Columns = months since signup.')
    display(ltv.round(2))

    # Payback analysis using breakeven_df
    meta_cac_full = summary['total_spend_usd'].sum() / summary['transacting_users'].sum() if summary['transacting_users'].sum() > 0 else 0
    print(f'\nMeta Ads CAC (full, campaign-10 cohort): ${meta_cac_full:.2f}')
    print()

    if breakeven_df is not None and not breakeven_df.empty:
        print('=== Payback period at current weighted CAC (ClientModel) ===')
        display(breakeven_df)

    # Heatmap chart
    fig_ltv = px.imshow(
        ltv.apply(pd.to_numeric, errors='coerce'),
        color_continuous_scale='RdYlGn',
        color_continuous_midpoint=0,
        title='Cohort LTV Heatmap — avg cumulative net USD per transacting user',
        labels={'x': 'Months since signup', 'y': 'Cohort month', 'color': 'Cum LTV (USD)'},
        aspect='auto',
    )
    fig_ltv.update_layout(height=400)
    fig_ltv.show()

    print("""
CONCLUSION A3:
- Green cells = cohort months with positive cumulative LTV (users have recouped KYC + COGS).
- Red cells = still underwater — card COGS and KYC cost dominate early months.
- The payback period tells you how long to wait before scaling — if >6 months, fix unit economics first.
- Watch the 'm0' column: high early revenue signals users who onramp immediately (best quality cohort).
""")
else:
    print('cohort_ltv data unavailable')


### A4 · Champion Client Profile (CPF Enrichment)

**Business question:** What demographic/financial profile do our highest-LTV (champion) users share?  
**Action:** Use champion attributes to build Meta Ads Lookalike Audience — target quality over volume.  
> Aggregate only — no individual CPF values are shown.


In [ ]:

# A4 · Champion Client Profile (CPF Enrichment)
# What demographic/financial profile do our highest-LTV (champion) users share?
# Use this to build Meta Ads custom/lookalike audiences.
# Aggregate only — no individual CPF values are shown.

from sqlalchemy import text as _text

_CPF_SQL = """
SELECT
    c.user_id,
    c.sexo                                                            AS gender,
    EXTRACT(YEAR FROM AGE(NOW(),
        TO_DATE(NULLIF(c.data_nascimento, ''), 'DD/MM/YYYY')))        AS age_years,
    c.raw_data->>'raw_faixa_poder_aquisitivo'                         AS purchasing_power_tier,
    (c.raw_data->>'raw_score')::FLOAT                                 AS credit_score,
    c.raw_data->>'raw_cbo'                                            AS cbo_code,
    c.raw_data->>'raw_escolaridade'                                   AS education_level
FROM cpf_validation_data c
"""

# Reuse the engine from the heatmap section (oq._engine_lazy)
with oq._engine_lazy.connect() as _conn:
    cpf_df = pd.read_sql(_text(_CPF_SQL), _conn)

# Join with RFM segments (user_id in segments_df is unmasked at this stage)
seg_for_join = segments_df[['user_id', 'segment']].copy() if segments_df is not None else pd.DataFrame()
cpf_seg = cpf_df.merge(seg_for_join, on='user_id', how='inner')

print(f'CPF records: {len(cpf_df):,}  |  Matched to segments: {len(cpf_seg):,}')
print()

def _profile(grp: pd.DataFrame, label: str) -> None:
    print(f'--- {label} (n={len(grp):,}) ---')
    if 'gender' in grp.columns:
        print('  Gender          :', grp['gender'].value_counts(normalize=True).map('{:.0%}'.format).to_dict())
    if 'age_years' in grp.columns:
        ages = grp['age_years'].dropna()
        if not ages.empty:
            print(f'  Age             : median={ages.median():.0f}  mean={ages.mean():.0f}  p25={ages.quantile(.25):.0f}  p75={ages.quantile(.75):.0f}')
    if 'purchasing_power_tier' in grp.columns:
        print('  Purch power tier:', grp['purchasing_power_tier'].value_counts(normalize=True).head(5).map('{:.0%}'.format).to_dict())
    if 'education_level' in grp.columns:
        print('  Education       :', grp['education_level'].value_counts(normalize=True).head(4).map('{:.0%}'.format).to_dict())
    if 'credit_score' in grp.columns:
        scores = grp['credit_score'].dropna()
        if not scores.empty:
            print(f'  Credit score    : median={scores.median():.0f}  mean={scores.mean():.0f}')
    print()

for seg_name in ['champion', 'active', 'at_risk', 'dormant']:
    sub = cpf_seg[cpf_seg['segment'] == seg_name]
    if not sub.empty:
        _profile(sub, seg_name.upper())

print("""
CONCLUSION A4:
- Compare 'champion' vs 'dormant' profiles — the delta reveals your ideal Meta Ads target.
- Feed champion attributes (age band, purchasing power tier, education) into Meta's
  Custom Audience + Lookalike Audience builder to acquire higher-quality users.
- Credit score and purchasing power tier are the strongest predictors of card activation.
- If champion users cluster in a specific occupation (CBO code), use LinkedIn-style interest
  targeting on Meta (finance professionals, crypto traders, international workers).
""")


### A5 · Product Cross-Sell Funnel

**Business question:** What % of users use 2+ products? Does the paid cohort cross-sell as well as organic?  
**Signal:** If meta_ads users are conversion-only, each has ~1/4 the LTV potential of a full-funnel user.


In [ ]:

# A5 · Product Cross-Sell Funnel
# How many users progress from 1 product to 2, 3, 4?
# Segment by acquisition_source for quality comparison.

if product_df is not None and segments_df is not None:
    prod = product_df.copy()
    # Re-attach acquisition_source from segments_df (same row order, user_id masked)
    seg_src = segments_df[['acquisition_source']].reset_index(drop=True)
    prod = pd.concat([prod.reset_index(drop=True), seg_src], axis=1)

    funnel_rows = []
    for src in sorted(prod['acquisition_source'].dropna().unique()):
        sub = prod[prod['acquisition_source'] == src]
        n = len(sub)
        funnel_rows.append({
            'source': src,
            'n_users': n,
            'has_conversion_%':  f"{sub['has_conversion'].mean():.0%}",
            'has_card_%':        f"{sub['has_card'].mean():.0%}",
            'has_swap_%':        f"{sub['has_swap'].mean():.0%}",
            'has_crossborder_%': f"{sub['has_crossborder'].mean():.0%}",
            '2+_products_%':     f"{(sub['n_products'] >= 2).mean():.0%}",
            '3+_products_%':     f"{(sub['n_products'] >= 3).mean():.0%}",
        })

    print('=== Product Adoption by Acquisition Source ===')
    display(pd.DataFrame(funnel_rows).to_string(index=False))

    # Overall distribution
    print('\n=== n_products distribution (all users) ===')
    print(product_df['n_products'].value_counts().sort_index().rename('users').to_frame().to_string())

    print("""
CONCLUSION A5:
- Users with 2+ products have significantly higher LTV (multiple revenue streams).
- If organic/founder users show higher multi-product rates than meta_ads cohort,
  paid acquisition is attracting weaker-fit users who use only one product.
- Cross-sell from conversion → card is the highest-leverage upsell to focus on.
""")
